# Everyday Data Science Demo: Predicting Bike Sharing Demand

**Lecture context:** Visual Studio Code, GitHub, GitHub Copilot, exploratory data analysis, and scikit-learn.

This notebook uses the public **UCI Bike Sharing Dataset** to answer an everyday question:

> Given the time of day and weather conditions, how many shared bikes might people rent?

The example is intentionally simple and practical. It demonstrates a complete workflow that students can run in VS Code, version with GitHub, and improve with GitHub Copilot or other VS Code coding agents.

## 1. How this demo connects to the lecture

This notebook is designed to be used as a single end-to-end class activity.

**Tools practiced**

- **VS Code:** open, edit, and run a Jupyter notebook.
- **GitHub:** create a repository, commit versions, push changes, and use issues or pull requests.
- **GitHub Copilot in VS Code:** ask for code explanations, generate helper functions, debug errors, and suggest visualizations.
- **Other VS Code agents/extensions:** ask for refactoring, documentation, tests, or alternative models.
- **EDA:** inspect data, detect patterns, and generate plots.
- **Machine learning:** train, evaluate, compare, and interpret predictive models using scikit-learn.

**Dataset source**

Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5W894.

## 2. Recommended repository structure

Create a GitHub repository such as:

```text
bike-sharing-eda-ml-demo/
├── notebooks/
│   └── bike_sharing_eda_ml_demo.ipynb
├── data/                 # ignored by Git if data are downloaded locally
├── README.md
├── requirements.txt
└── .gitignore
```

Suggested `.gitignore` entries:

```text
.ipynb_checkpoints/
__pycache__/
data/
*.pyc
.env
```

Suggested `requirements.txt`:

```text
pandas
numpy
matplotlib
scikit-learn
```

## 3. Setup

Run this notebook in VS Code using a Python environment that has `pandas`, `numpy`, `matplotlib`, and `scikit-learn` installed.

In VS Code, you can open the command palette and select **Python: Create Environment**. Then install packages with:

```bash
pip install pandas numpy matplotlib scikit-learn
```

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
from zipfile import ZipFile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 100)

## 4. Download and load the public dataset

The dataset contains hourly bike rental counts from the Capital Bikeshare system with weather and calendar variables.

The notebook tries two UCI URLs. If one URL changes, the other may still work.

In [ ]:
DATA_DIR = Path("data/bike_sharing")
DATA_DIR.mkdir(parents=True, exist_ok=True)

hour_csv = DATA_DIR / "hour.csv"

urls = [
    "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip",
]

if not hour_csv.exists():
    last_error = None
    for url in urls:
        try:
            print(f"Trying to download: {url}")
            req = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(req, timeout=30) as response:
                zip_bytes = response.read()
            with ZipFile(BytesIO(zip_bytes)) as zf:
                zf.extractall(DATA_DIR)
            print("Download and extraction complete.")
            break
        except Exception as exc:
            last_error = exc
            print(f"Failed: {exc}")
    else:
        raise RuntimeError("Could not download the dataset from the listed URLs.") from last_error
else:
    print("Dataset already exists locally.")

raw = pd.read_csv(hour_csv)
raw.head()

## 5. Understand the data

Important columns:

- `dteday`: date
- `season`: coded season
- `yr`: year, where 0 means 2011 and 1 means 2012
- `mnth`: month
- `hr`: hour of day
- `holiday`: whether the day is a holiday
- `weekday`: day of week
- `workingday`: 1 if the day is neither weekend nor holiday
- `weathersit`: weather category
- `temp`: normalized temperature
- `atemp`: normalized feeling temperature
- `hum`: normalized humidity
- `windspeed`: normalized wind speed
- `casual`: count of casual users
- `registered`: count of registered users
- `cnt`: total bike rentals, which is the prediction target

We should **not** use `casual` and `registered` as model features because they directly add up to `cnt`. In real prediction settings, those values would not be known in advance.

In [ ]:
print(f"Rows: {raw.shape[0]:,}")
print(f"Columns: {raw.shape[1]}")
raw.info()

In [ ]:
raw.describe().T

In [ ]:
missing = raw.isna().sum().sort_values(ascending=False)
missing[missing > 0]

## 6. Clean and engineer features

The UCI dataset stores weather variables in normalized form. Here we convert them back into more interpretable units using the dataset documentation.

- Temperature in Celsius: `temp_c = temp * (39 - (-8)) + (-8)`
- Feeling temperature in Celsius: `feels_like_c = atemp * (50 - (-16)) + (-16)`
- Humidity in percent: `humidity_pct = hum * 100`
- Windspeed in km/h: `windspeed_kmh = windspeed * 67`

In [ ]:
df = raw.copy()

df["date"] = pd.to_datetime(df["dteday"])
df["year"] = df["yr"].map({0: 2011, 1: 2012})

df["temp_c"] = df["temp"] * (39 - (-8)) + (-8)
df["feels_like_c"] = df["atemp"] * (50 - (-16)) + (-16)
df["humidity_pct"] = df["hum"] * 100
df["windspeed_kmh"] = df["windspeed"] * 67

df["season_label"] = df["season"].map({
    1: "winter",
    2: "spring",
    3: "summer",
    4: "fall",
})

df["weather_label"] = df["weathersit"].map({
    1: "clear_or_partly_cloudy",
    2: "mist_or_cloudy",
    3: "light_rain_or_snow",
    4: "heavy_rain_or_snow",
})

df["is_weekend"] = df["weekday"].isin([0, 6]).astype(int)
df["is_rush_hour"] = df["hr"].isin([7, 8, 16, 17, 18]).astype(int)

df[["date", "hr", "season_label", "weather_label", "temp_c", "humidity_pct", "windspeed_kmh", "cnt"]].head()

## 7. Exploratory Data Analysis

EDA helps us understand patterns before modeling. For this dataset, useful questions include:

1. When are bikes rented most often during the day?
2. How does demand change with weather?
3. Does temperature appear related to demand?
4. Which variables appear most correlated with rental counts?

In [ ]:
# Average demand by hour of day
hourly_demand = df.groupby("hr")["cnt"].mean()

plt.figure(figsize=(9, 5))
plt.plot(hourly_demand.index, hourly_demand.values, marker="o")
plt.title("Average Bike Rentals by Hour of Day")
plt.xlabel("Hour of Day")
plt.ylabel("Average Rentals")
plt.xticks(range(0, 24))
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Average demand by weather condition
weather_demand = df.groupby("weather_label")["cnt"].mean().sort_values()

plt.figure(figsize=(9, 5))
weather_demand.plot(kind="barh")
plt.title("Average Bike Rentals by Weather Condition")
plt.xlabel("Average Rentals")
plt.ylabel("Weather Condition")
plt.grid(True, axis="x", alpha=0.3)
plt.show()

In [ ]:
# Temperature versus demand
plt.figure(figsize=(9, 5))
plt.scatter(df["temp_c"], df["cnt"], alpha=0.15)
plt.title("Bike Rentals versus Temperature")
plt.xlabel("Temperature, Celsius")
plt.ylabel("Hourly Bike Rentals")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Correlation matrix for selected numeric variables
corr_cols = [
    "cnt", "hr", "mnth", "holiday", "weekday", "workingday",
    "temp_c", "feels_like_c", "humidity_pct", "windspeed_kmh", "is_weekend", "is_rush_hour"
]

corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr)
ax.set_xticks(np.arange(len(corr_cols)))
ax.set_yticks(np.arange(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=90)
ax.set_yticklabels(corr_cols)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

corr["cnt"].sort_values(ascending=False)

## 8. Machine learning problem framing

This is a **supervised regression** problem.

- **Supervised learning:** the training data include both inputs and known answers.
- **Regression:** the model predicts a continuous numeric value.
- **Target:** `cnt`, the number of bike rentals in one hour.
- **Features:** calendar variables, hour of day, weather category, temperature, humidity, and wind speed.

We use a time ordered split rather than a random split. The model trains on earlier observations and tests on later observations. This is closer to a real forecasting workflow.

In [ ]:
# Sort by time before splitting
model_df = df.sort_values(["date", "hr"]).reset_index(drop=True)

numeric_features = [
    "hr", "mnth", "temp_c", "feels_like_c", "humidity_pct", "windspeed_kmh",
    "is_weekend", "is_rush_hour"
]

categorical_features = [
    "season_label", "weather_label", "holiday", "weekday", "workingday"
]

features = numeric_features + categorical_features
target = "cnt"

X = model_df[features]
y = model_df[target]

split_index = int(len(model_df) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training rows: {len(X_train):,}")
print(f"Testing rows:  {len(X_test):,}")
print(f"Training period: {model_df.loc[:split_index-1, 'date'].min().date()} to {model_df.loc[:split_index-1, 'date'].max().date()}")
print(f"Testing period:  {model_df.loc[split_index:, 'date'].min().date()} to {model_df.loc[split_index:, 'date'].max().date()}")

## 9. Build scikit-learn pipelines

A scikit-learn `Pipeline` combines preprocessing and modeling into one reproducible workflow.

We compare four models:

1. **Dummy baseline:** predicts a simple average. A real model should beat this.
2. **Linear regression:** easy to explain, but may underfit nonlinear patterns.
3. **Random forest:** captures nonlinear interactions and variable importance.
4. **Histogram gradient boosting:** often strong for tabular data.

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

models = {
    "Dummy mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "Histogram gradient boosting": HistGradientBoostingRegressor(
        max_iter=200,
        learning_rate=0.05,
        random_state=42,
    ),
}

fitted_models = {}
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model),
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    
    fitted_models[name] = pipe
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df

In [ ]:
# Visual comparison of model performance
plt.figure(figsize=(9, 5))
plt.barh(results_df["Model"], results_df["RMSE"])
plt.title("Model Comparison, Lower RMSE is Better")
plt.xlabel("Root Mean Squared Error")
plt.ylabel("Model")
plt.grid(True, axis="x", alpha=0.3)
plt.show()

## 10. Inspect predictions from the best model

The next cell selects the model with the lowest RMSE and plots actual versus predicted bike rentals for the first week of the test period.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]

print(f"Best model: {best_name}")

test_predictions = best_model.predict(X_test)

test_plot = model_df.iloc[split_index:].copy()
test_plot["prediction"] = test_predictions

sample = test_plot.iloc[:24 * 7].copy()
sample["timestamp"] = sample["date"] + pd.to_timedelta(sample["hr"], unit="h")

plt.figure(figsize=(12, 5))
plt.plot(sample["timestamp"], sample["cnt"], label="Actual")
plt.plot(sample["timestamp"], sample["prediction"], label="Predicted")
plt.title(f"Actual versus Predicted Bike Rentals, First Test Week, {best_name}")
plt.xlabel("Date and Time")
plt.ylabel("Hourly Bike Rentals")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Interpret the random forest model

Interpretability is important in engineering and planning contexts. Here we inspect feature importance from the random forest model.

Feature importance is not the same as causation. It only shows which variables the model used more strongly to reduce prediction error.

In [ ]:
rf_pipe = fitted_models["Random forest"]
feature_names = rf_pipe.named_steps["preprocess"].get_feature_names_out()
importances = rf_pipe.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
)

importance_df

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(importance_df["feature"][::-1], importance_df["importance"][::-1])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Try a simple everyday prediction

Suppose the city wants to estimate expected bike rentals for a pleasant spring weekday morning commute.

Change the values below and rerun the cell. This is a good place to ask Copilot:

> Create three different weather scenarios and compare predicted bike rental demand.

In [ ]:
example = pd.DataFrame([
    {
        "hr": 8,
        "mnth": 5,
        "temp_c": 22,
        "feels_like_c": 22,
        "humidity_pct": 55,
        "windspeed_kmh": 10,
        "is_weekend": 0,
        "is_rush_hour": 1,
        "season_label": "spring",
        "weather_label": "clear_or_partly_cloudy",
        "holiday": 0,
        "weekday": 2,
        "workingday": 1,
    }
])

predicted_rentals = best_model.predict(example)[0]
print(f"Predicted bike rentals for this hour: {predicted_rentals:.0f}")

## 13. Optional extension: convert the problem into classification

Machine learning task types include:

- **Regression:** predict a numeric value, such as hourly bike rentals.
- **Classification:** predict a category, such as high demand versus normal demand.
- **Clustering:** find natural groups without known labels.
- **Dimensionality reduction:** compress many variables into fewer informative dimensions.
- **Reinforcement learning:** learn decisions through feedback and rewards.

Below is a small classification setup. It labels the top 25 percent of rental hours as high demand. Students can complete this as an exercise.

In [ ]:
classification_df = model_df.copy()
threshold = classification_df["cnt"].quantile(0.75)
classification_df["high_demand"] = (classification_df["cnt"] >= threshold).astype(int)

print(f"High-demand threshold: {threshold:.0f} rentals per hour")
classification_df[["date", "hr", "cnt", "high_demand"]].head()

## 14. VS Code, GitHub, and Copilot practice tasks

Use this notebook to practice a realistic data science workflow.

### GitHub tasks

1. Create a repository named `bike-sharing-eda-ml-demo`.
2. Add this notebook under `notebooks/`.
3. Add a `README.md` explaining the project question, dataset, and model results.
4. Commit after each major section:
   - `Add dataset download and loading`
   - `Add EDA charts`
   - `Add scikit-learn models`
   - `Add interpretation and prediction example`
5. Create a branch named `classification-extension` and implement the optional classification task.
6. Open a pull request from `classification-extension` into `main`.

### Copilot or coding-agent prompts to try

- Explain this notebook as if I am new to Python data science.
- Add comments to the model training code.
- Refactor the repeated plotting code into reusable functions.
- Add a confusion matrix for the optional classification task.
- Create a short README for this repository.
- Suggest civil engineering applications that use the same workflow.

### Discussion questions

- What variables appear most useful for predicting bike demand?
- Why should `casual` and `registered` not be used as features?
- Why might a time ordered split be better than a random split here?
- Which model is most accurate? Which model is easiest to explain?
- How would this workflow change for bus ridership, traffic crashes, pavement condition, flooding, or building energy use?

## 15. Civil engineering applications using the same workflow

The same data science pattern applies to many civil engineering problems.

| Domain | Example prediction target | Typical data sources |
|---|---|---|
| Transportation | Hourly ridership, traffic volume, bike demand, crash risk | GTFS, APC, loop detectors, GPS, weather, open data portals |
| Structural engineering | Bridge condition rating or inspection priority | Inspection records, sensor data, age, traffic loading, environment |
| Construction engineering | Cost overrun risk or schedule delay | Project logs, weather, labor records, progress photos, BIM metadata |
| Water resources | Flood depth, runoff, water demand | Rainfall, land cover, terrain, stream gauges, hydraulic models |
| Geotechnical engineering | Settlement, slope failure susceptibility | Soil properties, rainfall, topography, instrumentation |
| Urban analytics | Accessibility, demand hotspots, service gaps | Census data, POIs, transit feeds, mobility traces, land use |

The core workflow is consistent:

1. Define the engineering question.
2. Load and inspect the data.
3. Clean, join, and engineer features.
4. Explore patterns visually.
5. Train baseline and candidate models.
6. Evaluate with appropriate metrics.
7. Interpret results for planning or operational decisions.
8. Version the work in GitHub and document assumptions.

## 16. Suggested student deliverable

Submit a GitHub repository with:

1. This completed notebook.
2. A `README.md` with the project question, dataset description, model comparison table, and one interpretation figure.
3. At least four commits with meaningful messages.
4. One issue describing a possible model improvement.
5. One pull request implementing a small improvement.

A strong submission should show not only code that runs, but also clear reasoning about data, model assumptions, and practical use.